In [4]:
%pip install toml

Defaulting to user installation because normal site-packages is not writeable
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
Using cached toml-0.10.2-py2.py3-none-any.whl (16 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\hisuk\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
import pyodbc
import pandas as pd

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost\\SQLEXPRESS;"
    "DATABASE=AdventureWorks2025;"
    "Trusted_Connection=yes;"
)

query = """
SELECT
    sod.SalesOrderID,
    sod.SalesOrderDetailID,
    sod.ProductID,
    sod.OrderQty,
    sod.UnitPrice,
    sod.UnitPriceDiscount,
    sod.LineTotal,
    sod.ModifiedDate,
    p.Name AS ProductName,
    p.ProductNumber,
    p.Color,
    p.Class,
    p.Style
FROM Sales.SalesOrderDetail sod
LEFT JOIN Production.Product p
    ON sod.ProductID = p.ProductID
"""

df = pd.read_sql(query, conn)
df.head()
print(df.shape)

C:\Users\hisuk\AppData\Local\Temp\ipykernel_19716\2132472811.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


(121317, 13)


In [6]:
import sys
!{sys.executable} -m pip install requests

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\hisuk\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Product Name Text Preprocessing

Standardized product names by converting them to lowercase and handling missing values to ensure consistent text matching.

In [11]:
import sys
from pathlib import Path
import requests

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from config.config import load_config

cfg = load_config()

url = cfg["api"]["base_url"]
api_key = cfg["api"]["api_key"]

headers = {
    "apy-token": api_key,
    "Content-Type": "application/json"
}

payload = {
    "content": "Razer Blade 16 Gaming Laptop RTX4090 i9 32GB RAM 2TB SSD Windows 11",
    "language": "English"
}

response = requests.post(url, json=payload, headers=headers)
print(response.status_code, response.text)

429 {
  "message":"API rate limit exceeded"
}


In [ ]:
import requests
import time

response = requests.post(url, json=payload, headers=headers)
print("POST:", response.status_code, response.text)


In [ ]:
print(response.headers)

In [ ]:
import requests
import time

MAX_RETRY = 3

job_id = None

for attempt in range(MAX_RETRY):
    response = requests.post(url, json=payload, headers=headers)
    print("POST:", response.status_code, response.text)

    if response.status_code == 429:
        print("Rate limit → retry after 20 seconds")
        time.sleep(20)
        continue

    response.raise_for_status()

    post_data = response.json()
    job_id = post_data.get("job_id")

    if not job_id:
        raise Exception(f"job_id not found: {post_data}")

    break

if not job_id:
    raise Exception("POST failed: retry exceeded")